# 日级特征

STEP 1 质量过滤 → STEP 2 时间窗口 → STEP 3 日级聚合 → STEP 4 合并 → STEP 5 缺失/mask → STEP 6 滚动个体归一化→ STEP 7 输出。

In [21]:
import pandas as pd
import numpy as np
from pathlib import Path

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "data_process" else cwd
SUB = ROOT / "subdataset"
OUT = ROOT / "processed_data"
OUT.mkdir(parents=True, exist_ok=True)

cycle = pd.read_csv(SUB / "cycle_anchor_clean.csv")
cycle_days = cycle[["id", "study_interval", "day_in_study"]].drop_duplicates()
print("cycle_days:", len(cycle_days))

cycle_days: 2797


**时间约定（据 mcPHASES 文档）**：`day_in_study` = 从研究开始的天数索引（日历日）；`timestamp` = 当日时间 `HH:MM:SS`（本地时间）。故 `(id, study_interval, day_in_study, timestamp)` 表示该日历日该时刻的一条记录。睡眠边界来自 computed_temperature 的 `sleep_start/end_day_in_study` + `sleep_start/end_timestamp`，一夜归属到 **sleep_end_day_in_study**（醒来日）。

In [22]:
def ts_to_min(ts):
    """'HH:MM:SS' -> minutes since midnight (0~1440)."""
    if pd.isna(ts):
        return np.nan
    p = str(ts).strip().split(":")
    h = int(p[0]) if len(p) > 0 else 0
    m = int(float(p[1])) if len(p) > 1 else 0
    s = float(p[2]) if len(p) > 2 else 0
    return h * 60 + m + s / 60

# Sleep bounds: one row per night attributed to sleep_end_day_in_study; rename to day_in_study to avoid duplicate column
ct = pd.read_csv(SUB / "computed_temperature_cycle.csv")
ct_night = ct.groupby(["id", "study_interval", "sleep_end_day_in_study"], as_index=False).first()
ct_night["sleep_start_min"] = ct_night["sleep_start_timestamp"].map(ts_to_min)
ct_night["sleep_end_min"] = ct_night["sleep_end_timestamp"].map(ts_to_min)
sleep_bounds = ct_night[["id", "study_interval", "sleep_end_day_in_study", "sleep_start_min", "sleep_end_min"]].rename(
    columns={"sleep_end_day_in_study": "day_in_study"}
)
print("sleep_bounds rows:", len(sleep_bounds))

sleep_bounds rows: 2654


In [23]:
# STEP 1：HRV 质量过滤
hrv = pd.read_csv(SUB / "heart_rate_variability_details_cycle.csv")
n_hrv = len(hrv)
hrv = hrv[(hrv["coverage"] >= 0.6) & (hrv["rmssd"] > 0) & (hrv["low_frequency"] > 0) & (hrv["high_frequency"] > 0)]
hrv["timestamp_min"] = hrv["timestamp"].map(ts_to_min)
print("HRV: %.2f%% kept" % (100 * len(hrv) / n_hrv))

HRV: 99.95% kept


In [24]:
def agg_hrv(df):
    g = df.groupby(["id", "study_interval", "day_in_study"], as_index=False)
    lf = g["low_frequency"].mean()
    hf = g["high_frequency"].mean()
    out = g.agg(rmssd_mean=("rmssd", "mean")).merge(lf.rename(columns={"low_frequency": "lf_mean"}))
    out = out.merge(hf.rename(columns={"high_frequency": "hf_mean"}))
    out["lf_hf_ratio"] = out["lf_mean"] / (out["hf_mean"].replace(0, np.nan))
    return out[["id", "study_interval", "day_in_study", "rmssd_mean", "lf_mean", "hf_mean", "lf_hf_ratio"]]

hrv_merged = hrv.merge(sleep_bounds, on=["id", "study_interval", "day_in_study"], how="left")
hrv_full = agg_hrv(hrv)
# sleep: same-day time<=sleep_end + prev-day time>=sleep_start -> attribute to wake_day
hrv_same = hrv_merged[hrv_merged["timestamp_min"] <= hrv_merged["sleep_end_min"]][hrv.columns]
hrv_prev = hrv.copy()
hrv_prev["wake_day"] = hrv_prev["day_in_study"] + 1
hrv_prev = hrv_prev.merge(sleep_bounds.rename(columns={"day_in_study": "wake_day"})[["id", "study_interval", "wake_day", "sleep_start_min"]], on=["id", "study_interval", "wake_day"], how="inner")
hrv_prev = hrv_prev[hrv_prev["timestamp_min"] >= hrv_prev["sleep_start_min"]].drop(columns=["sleep_start_min"], errors="ignore")
hrv_prev["day_in_study"] = hrv_prev["wake_day"]
hrv_prev = hrv_prev[hrv.columns]
hrv_sleep = agg_hrv(pd.concat([hrv_same, hrv_prev], ignore_index=True))
# morning: same-day [sleep_end, sleep_end+30]
hrv_morning = agg_hrv(hrv_merged[(hrv_merged["timestamp_min"] >= hrv_merged["sleep_end_min"]) & (hrv_merged["timestamp_min"] < hrv_merged["sleep_end_min"] + 30)][hrv.columns])
# evening: 30min before sleep -> wake_day. If sleep_start < 30min (early morning), span days: prev [start_prev,1440) + same [0,sleep_start_min]
sb_ev = sleep_bounds[["id", "study_interval", "day_in_study", "sleep_start_min"]].rename(columns={"day_in_study": "wake_day"})
hrv_ev_prev = hrv.copy()
hrv_ev_prev["wake_day"] = hrv_ev_prev["day_in_study"] + 1
hrv_ev_prev = hrv_ev_prev.merge(sb_ev, on=["id", "study_interval", "wake_day"], how="inner")
start_prev = (hrv_ev_prev["sleep_start_min"] - 30 + 1440) % 1440
mask_prev = (hrv_ev_prev["sleep_start_min"] >= 30) & (hrv_ev_prev["timestamp_min"] >= hrv_ev_prev["sleep_start_min"] - 30) & (hrv_ev_prev["timestamp_min"] < hrv_ev_prev["sleep_start_min"])
mask_prev = mask_prev | ((hrv_ev_prev["sleep_start_min"] < 30) & (hrv_ev_prev["timestamp_min"] >= start_prev))
hrv_ev_prev = hrv_ev_prev.loc[mask_prev].copy()
hrv_ev_prev["day_in_study"] = hrv_ev_prev["wake_day"]
hrv_ev_prev = hrv_ev_prev[hrv.columns]
hrv_ev_same = hrv.merge(sleep_bounds[["id", "study_interval", "day_in_study", "sleep_start_min"]], on=["id", "study_interval", "day_in_study"], how="inner")
hrv_ev_same = hrv_ev_same[(hrv_ev_same["sleep_start_min"] < 30) & (hrv_ev_same["timestamp_min"] < hrv_ev_same["sleep_start_min"])][hrv.columns]
hrv_evening = agg_hrv(pd.concat([hrv_ev_prev, hrv_ev_same], ignore_index=True))
print("HRV daily: full", len(hrv_full), "sleep", len(hrv_sleep), "morning", len(hrv_morning), "evening", len(hrv_evening))

HRV daily: full 2468 sleep 2411 morning 7 evening 1003


In [25]:
def agg_hr_window(path, window, sleep_bounds_df, cycle_days_df, ts_to_min_fn, key=None):
    key = key or ["id", "study_interval", "day_in_study"]
    sb = sleep_bounds_df
    agg_list = []
    for chunk in pd.read_csv(path, chunksize=500_000):
        chunk = chunk[(chunk["bpm"] >= 30) & (chunk["bpm"] <= 220) & (chunk["confidence"] >= 0.5)]
        chunk["timestamp_min"] = chunk["timestamp"].map(ts_to_min_fn)
        chunk = chunk.merge(cycle_days_df, on=key, how="inner")
        if window == "full":
            use = chunk[key + ["bpm"]]
        else:
            chunk = chunk.merge(sb, on=key, how="left")
            if window == "sleep":
                same = chunk[chunk["timestamp_min"] <= chunk["sleep_end_min"]][key + ["bpm"]]
                prev = chunk.drop(columns=["sleep_start_min", "sleep_end_min"], errors="ignore").copy()
                prev["wake_day"] = prev["day_in_study"] + 1
                prev = prev.merge(sb[["id", "study_interval", "day_in_study", "sleep_start_min"]].rename(columns={"day_in_study": "wake_day"}), on=["id", "study_interval", "wake_day"], how="inner")
                prev = prev[prev["timestamp_min"] >= prev["sleep_start_min"]].copy()
                prev["day_in_study"] = prev["wake_day"]
                prev = prev[key + ["bpm"]]
                use = pd.concat([same, prev], ignore_index=True)
            elif window == "morning":
                use = chunk[(chunk["timestamp_min"] >= chunk["sleep_end_min"]) & (chunk["timestamp_min"] < chunk["sleep_end_min"] + 30)][key + ["bpm"]]
            elif window == "evening":
                # Prev day: need this night's sleep_start (attr to wake_day); same-day early: need this night's sleep_start (attr to day_in_study)
                # Drop sleep_* from first merge to avoid _x/_y duplicate cols in second merge
                chunk = chunk.drop(columns=["sleep_start_min", "sleep_end_min"], errors="ignore")
                chunk["wake_day"] = chunk["day_in_study"] + 1
                chunk = chunk.merge(sb[["id", "study_interval", "day_in_study", "sleep_start_min"]].rename(columns={"day_in_study": "wake_day"}), on=["id", "study_interval", "wake_day"], how="inner")
                start_prev = (chunk["sleep_start_min"] - 30 + 1440) % 1440
                m_prev = (chunk["day_in_study"] == chunk["wake_day"] - 1) & (((chunk["sleep_start_min"] >= 30) & (chunk["timestamp_min"] >= chunk["sleep_start_min"] - 30) & (chunk["timestamp_min"] < chunk["sleep_start_min"])) | ((chunk["sleep_start_min"] < 30) & (chunk["timestamp_min"] >= start_prev)))
                use_prev = chunk.loc[m_prev, key + ["bpm"]].copy()
                use_prev["day_in_study"] = chunk.loc[m_prev, "wake_day"].values
                same_df = chunk[key + ["timestamp_min", "bpm"]].merge(sb[["id", "study_interval", "day_in_study", "sleep_start_min"]], on=key, how="inner")
                m_same = (same_df["sleep_start_min"] < 30) & (same_df["timestamp_min"] < same_df["sleep_start_min"])
                use_same = same_df.loc[m_same, key + ["bpm"]].copy()
                use = pd.concat([use_prev, use_same], ignore_index=True)
            else:
                use = chunk[key + ["bpm"]]
        if len(use) == 0:
            continue
        agg_list.append(use.groupby(key, as_index=False)["bpm"].agg(hr_mean="mean", hr_std="std", hr_min="min", hr_max="max"))
    if not agg_list:
        return pd.DataFrame(columns=key + ["hr_mean", "hr_std", "hr_min", "hr_max"])
    return pd.concat(agg_list).groupby(key, as_index=False).agg(hr_mean=("hr_mean", "mean"), hr_std=("hr_std", "mean"), hr_min=("hr_min", "min"), hr_max=("hr_max", "max"))

In [26]:
def agg_wt_window(path, window, sleep_bounds_df, cycle_days_df, ts_to_min_fn, key=None):
    key = key or ["id", "study_interval", "day_in_study"]
    sb = sleep_bounds_df
    col = "temperature_diff_from_baseline"
    agg_list = []
    for chunk in pd.read_csv(path, chunksize=500_000):
        chunk = chunk[chunk[col].abs() <= 5]
        chunk["timestamp_min"] = chunk["timestamp"].map(ts_to_min_fn)
        chunk = chunk.merge(cycle_days_df, on=key, how="inner")
        if window == "full":
            use = chunk[key + [col]]
        else:
            chunk = chunk.merge(sb, on=key, how="left")
            if window == "sleep":
                same = chunk[chunk["timestamp_min"] <= chunk["sleep_end_min"]][key + [col]]
                prev = chunk.drop(columns=["sleep_start_min", "sleep_end_min"], errors="ignore").copy()
                prev["wake_day"] = prev["day_in_study"] + 1
                prev = prev.merge(sb[["id", "study_interval", "day_in_study", "sleep_start_min"]].rename(columns={"day_in_study": "wake_day"}), on=["id", "study_interval", "wake_day"], how="inner")
                prev = prev[prev["timestamp_min"] >= prev["sleep_start_min"]].copy()
                prev["day_in_study"] = prev["wake_day"]
                prev = prev[key + [col]]
                use = pd.concat([same, prev], ignore_index=True)
            elif window == "morning":
                use = chunk[(chunk["timestamp_min"] >= chunk["sleep_end_min"]) & (chunk["timestamp_min"] < chunk["sleep_end_min"] + 30)][key + [col]]
            elif window == "evening":
                chunk = chunk.drop(columns=["sleep_start_min", "sleep_end_min"], errors="ignore")
                chunk["wake_day"] = chunk["day_in_study"] + 1
                chunk = chunk.merge(sb[["id", "study_interval", "day_in_study", "sleep_start_min"]].rename(columns={"day_in_study": "wake_day"}), on=["id", "study_interval", "wake_day"], how="inner")
                start_prev = (chunk["sleep_start_min"] - 30 + 1440) % 1440
                m_prev = (chunk["day_in_study"] == chunk["wake_day"] - 1) & (((chunk["sleep_start_min"] >= 30) & (chunk["timestamp_min"] >= chunk["sleep_start_min"] - 30) & (chunk["timestamp_min"] < chunk["sleep_start_min"])) | ((chunk["sleep_start_min"] < 30) & (chunk["timestamp_min"] >= start_prev)))
                use_prev = chunk.loc[m_prev, key + [col]].copy()
                use_prev["day_in_study"] = chunk.loc[m_prev, "wake_day"].values
                same_df = chunk[key + ["timestamp_min", col]].merge(sb[["id", "study_interval", "day_in_study", "sleep_start_min"]], on=key, how="inner")
                m_same = (same_df["sleep_start_min"] < 30) & (same_df["timestamp_min"] < same_df["sleep_start_min"])
                use_same = same_df.loc[m_same, key + [col]].copy()
                use = pd.concat([use_prev, use_same], ignore_index=True)
            else:
                use = chunk[key + [col]]
        if len(use) == 0:
            continue
        agg_list.append(use.groupby(key, as_index=False)[col].agg(wt_mean="mean", wt_std="std", wt_min="min", wt_max="max"))
    if not agg_list:
        return pd.DataFrame(columns=key + ["wt_mean", "wt_std", "wt_min", "wt_max"])
    return pd.concat(agg_list).groupby(key, as_index=False).agg(wt_mean=("wt_mean", "mean"), wt_std=("wt_std", "mean"), wt_min=("wt_min", "min"), wt_max=("wt_max", "max"))

In [27]:
# Daily: nightly_temperature (by day_in_study), resting_hr
ct_daily = ct_night[["id", "study_interval", "sleep_end_day_in_study", "nightly_temperature"]].rename(
    columns={"sleep_end_day_in_study": "day_in_study"}
)
rhr = pd.read_csv(SUB / "resting_heart_rate_cycle.csv")[["id", "study_interval", "day_in_study", "value"]].rename(columns={"value": "resting_hr"})
hrv_by_w = {"full": hrv_full, "sleep": hrv_sleep, "morning": hrv_morning, "evening": hrv_evening}
key = ["id", "study_interval", "day_in_study"]

In [28]:
# STEP 3+4: Aggregate HR/WT by window, merge with HRV, nightly_temperature, resting_hr; cycle_days as base
hr_path = SUB / "heart_rate_cycle.csv"
wt_path = SUB / "wrist_temperature_cycle.csv"
daily_by_window = {}
for w in ["full", "sleep", "morning", "evening"]:
    hr_w = agg_hr_window(hr_path, w, sleep_bounds, cycle_days, ts_to_min) if hr_path.exists() else pd.DataFrame(columns=key + ["hr_mean", "hr_std", "hr_min", "hr_max"])
    wt_w = agg_wt_window(wt_path, w, sleep_bounds, cycle_days, ts_to_min) if wt_path.exists() else pd.DataFrame(columns=key + ["wt_mean", "wt_std", "wt_min", "wt_max"])
    base = cycle_days.merge(hrv_by_w[w], on=key, how="left").merge(hr_w, on=key, how="left").merge(wt_w, on=key, how="left").merge(ct_daily, on=key, how="left").merge(rhr, on=key, how="left")
    daily_by_window[w] = base
    print(w, "rows", len(base))

full rows 6737
sleep rows 6737
morning rows 6737
evening rows 6737


In [29]:
# STEP 5: Missing value handling - fill with subject mean by (id, study_interval) and set mask
feat_cols = [c for c in daily_by_window["full"].columns if c not in key and c != "day_in_study" and pd.api.types.is_numeric_dtype(daily_by_window["full"][c])]
for w, df in daily_by_window.items():
    out = df.copy()
    for col in feat_cols:
        if col not in out.columns:
            continue
        miss = out[col].isna()
        out[col + "_missing"] = miss.astype(int)
        subj_mean = out.groupby(["id", "study_interval"])[col].transform("mean")
        out.loc[miss, col] = subj_mean
        out.loc[out[col].isna(), col] = 0
    daily_by_window[w] = out
print("feat_cols:", feat_cols)

feat_cols: ['rmssd_mean', 'lf_mean', 'hf_mean', 'lf_hf_ratio', 'hr_mean', 'hr_std', 'hr_min', 'hr_max', 'wt_mean', 'wt_std', 'wt_min', 'wt_max', 'nightly_temperature', 'resting_hr']


In [30]:
# STEP 6: Within-individual rolling normalisation (approach B): μ_t, σ_t from past K days only; z_t = (x_t - μ_t) / (σ_t + ε)
K_ROLL = 14
EPS = 1e-6
norm_cols = [c for c in feat_cols if not c.endswith("_missing")]

def rolling_z(df, cols, k, eps):
    out = df.sort_values(["id", "study_interval", "day_in_study"]).copy()
    for col in cols:
        if col not in out.columns:
            continue
        z = np.full(len(out), np.nan)
        for (iid, sid), grp in out.groupby(["id", "study_interval"]):
            x = grp[col].values
            idx = grp.index
            for i in range(len(x)):
                hist = x[:i]
                if len(hist) == 0:
                    z[idx[i]] = 0
                    continue
                hist_k = hist[-k:]
                mu = np.nanmean(hist_k)
                sig = np.nanstd(hist_k)
                if np.isnan(sig) or sig < eps:
                    sig = eps
                z[idx[i]] = (x[i] - mu) / sig
        out[col + "_z"] = z
    return out

for w in daily_by_window:
    daily_by_window[w] = rolling_z(daily_by_window[w], norm_cols, K_ROLL, EPS)
print("rolling z done")

rolling z done


In [31]:
# STEP 7: Output 4-window daily feature tables + index.csv
cycle_days.to_csv(OUT / "index.csv", index=False)
for w, name in [("full", "full"), ("sleep", "sleep"), ("morning", "morning"), ("evening", "evening")]:
    daily_by_window[w].to_csv(OUT / f"{name}.csv", index=False)
print("saved:", list(OUT.glob("*.csv")))

saved: [PosixPath('/Users/xujing/FYP/main_workspace/processed_data/full.csv'), PosixPath('/Users/xujing/FYP/main_workspace/processed_data/evening.csv'), PosixPath('/Users/xujing/FYP/main_workspace/processed_data/morning.csv'), PosixPath('/Users/xujing/FYP/main_workspace/processed_data/sleep.csv'), PosixPath('/Users/xujing/FYP/main_workspace/processed_data/index.csv')]


In [32]:
# Check: len(X) == len(index); model input uses _z columns
idx = pd.read_csv(OUT / "index.csv")
full_df = pd.read_csv(OUT / "full.csv")
z_cols = [c for c in full_df.columns if c.endswith("_z")]
X = full_df[z_cols].values if z_cols else full_df[[c for c in full_df.columns if c not in key and not c.endswith("_missing")]].values
print("index rows:", len(idx), "X shape:", X.shape, "len(X)==len(index):", len(X) == len(idx))

index rows: 2797 X shape: (6737, 14) len(X)==len(index): False
